# Stage 08 — Reporting

**Owner:** Pujan Dey (Visualisation & Reporting Lead)

**Objective.** Consolidate everything the presentation needs:
- A single metrics summary table (RQ1 regression + RQ2 classification).
- The paired-CV t-test results (RQ1 OLS vs Ridge, RQ2 RF vs NB).
- Listing of every figure produced in `outputs/figures/`.
- A short narrative template linking findings back to Assessment 1 stakeholders.

**Inputs.** Checkpoints 05, 06, 07.

**Outputs.**
- `outputs/tables/metrics_summary.csv`
- `outputs/tables/paired_ttest_results.json` (also written by notebook 07)

## 1. Setup and load all checkpoints

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import config, reporting

reg_metrics = json.load(open(config.CHECKPOINT_DIR / "05_reg_metrics.json"))
cls_metrics = json.load(open(config.CHECKPOINT_DIR / "06_cls_metrics.json"))
ttest_results = json.load(open(config.CHECKPOINT_DIR / "07_ttest_results.json"))

rf_metrics = cls_metrics["random_forest"]
nb_metrics = cls_metrics["naive_bayes"]
print("Loaded regression metrics, classification metrics, and t-test results.")

## 2. Consolidated metrics summary

In [ ]:
path = reporting.write_metrics_summary(reg_metrics, rf_metrics, nb_metrics)
print("Written →", path)
pd.read_csv(path)

## 3. Paired t-test table (RQ1 + RQ2)

In [ ]:
rows = []
for key, label in [("rq1_ols_vs_ridge", "RQ1 — OLS vs Ridge (R²)"),
                   ("rq2_rf_vs_nb",     "RQ2 — RandomForest vs NaiveBayes (accuracy)")]:
    r = ttest_results[key]
    rows.append({
        "comparison": label,
        "mean_a (model_a)": f"{r['mean_a']:.4f} ({r['model_a']})",
        "mean_b (model_b)": f"{r['mean_b']:.4f} ({r['model_b']})",
        "t_statistic": round(r["t_statistic"], 3),
        "p_value": round(r["p_value"], 4),
        "significant_at_0.05": r["significant_at_0.05"],
        "better_model": r["better_model"],
    })
pd.DataFrame(rows)

## 4. Figure inventory for the slides

In [ ]:
figs = sorted(config.FIG_DIR.glob("*.png"))
pd.DataFrame({"figure": [f.name for f in figs],
              "size_kb": [round(f.stat().st_size / 1024, 1) for f in figs]})

## 5. Narrative template for the live presentation

Use the fields below to draft each slide. Keep quantitative claims tied to the loaded metrics rather than rounded from memory.

### Slide A — problem recap (Assessment 1 §1.2, §1.3)
Gender inequality persists in Australian workplaces (21–22% pay gap, under-representation in leadership). WGEA needs evidence-based levers. Our analysis answers four questions on the 2024-25 compliance cohort.

### Slide B — pipeline flowchart
8 WGEA CSVs → employer master → policy flags → external ABS integration → encoded features → RQ1 regression + RQ2 classification + RQ3/RQ4 EDA.

### Slide C — RQ1 findings
- Adjusted $R^2$ = `{reg_metrics['Adj_R2']:.3f}` ; RMSE = `{reg_metrics['RMSE']:.3f}`.
- Drivers: top positive / negative coefficients from `rq1_coefficients.png`.
- Assumption checks: `rq1_residuals_vs_fitted.png` + `rq1_qq_plot.png`.

### Slide D — RQ2 findings
- Random Forest accuracy = `{rf_metrics['Accuracy']:.3f}` ; Naive Bayes accuracy = `{nb_metrics['Accuracy']:.3f}`.
- Paired-CV t-test → p = `{ttest_results['rq2_rf_vs_nb']['p_value']:.4f}` — {conclusion about H0}.
- Top drivers from `rq2_feature_importance.png`.

### Slide E — RQ3/RQ4 visual story
- Industry + size patterns from `rq3_women_share_by_division.png`, `rq3_women_share_by_size.png`.
- Policy lifts from `rq4_policy_vs_women.png`.

### Slide F — limitations (Assessment 1 §2.4)
Self-report bias, categorical employer size, aggregated employer-level only, public-sector excluded.

### Slide G — stakeholder implications (Assessment 1 §1.4)
Policymakers, employers/HR, employees/advocacy groups, researchers.

## Summary

All artefacts needed for the live presentation are now in `outputs/figures/` and `outputs/tables/`. The slide deck can be built directly from these files.